# Unidad 5 — Inferencia

Toda la materia hasta acá nos permitía **calcular probabilidades** dados los parámetros.
La inferencia invierte el problema: vemos **una muestra** y queremos hacer afirmaciones
sobre los **parámetros** que la generaron.

Tres herramientas centrales:

- **Intervalos de confianza** (IC) — rango plausible para $\mu$, $p$ o $\sigma^2$.
- **Pruebas de hipótesis** — decisión binaria con control del error tipo I.
- **Tamaño muestral** — cuánto necesitamos para una precisión dada.


## Importaciones

In [1]:
import numpy as np
import pandas as pd

from core import Observations, Settings
from exercises import (
    IntervalContainsInput,
    NumericAnswerInput,
    verify_interval_contains,
    verify_numeric_answer,
)
from inference import (
    MeanKnownVarianceInput,
    MeanUnknownVarianceInput,
    OneSampleMeanTestInput,
    ProportionInput,
    SampleSizeForMeanInput,
    SampleSizeForProportionInput,
    VarianceInput,
    build_confidence_interval_for_mean_known_variance,
    build_confidence_interval_for_mean_unknown_variance,
    build_confidence_interval_for_proportion,
    build_confidence_interval_for_variance,
    sample_size_for_mean,
    sample_size_for_proportion,
    test_one_sample_mean,
)
from inference.hypothesis_tests import Alternative
from sampling import BootstrapInput, bootstrap_mean
from visualization import BootstrapDistributionChartInput, chart_bootstrap_distribution
from widgets import MeanCIExplorerInput, build_mean_ci_explorer

In [2]:
settings = Settings()

## Concreto: IC para la media con $\sigma$ conocido

Suponemos que tenemos $n = 36$ mediciones de tiempo de servicio con $\bar{x} = 12$ min y
sabemos que $\sigma = 3$ min (info histórica). Queremos un IC al 95%.

$$ \bar{x} \pm z_{1-\alpha/2}\,\frac{\sigma}{\sqrt{n}} = 12 \pm 1{,}96 \cdot \frac{3}{6}
 = 12 \pm 0{,}98 = (11{,}02,\ 12{,}98). $$


In [3]:
confidence_interval = build_confidence_interval_for_mean_known_variance(
    MeanKnownVarianceInput(
        sample_mean=12.0,
        population_standard_deviation=3.0,
        sample_size=36,
        confidence_level=0.95,
    )
)
confidence_interval

MeanConfidenceInterval(point_estimate=12.0, lower_bound=11.020018007729973, upper_bound=12.979981992270027, margin_of_error=0.979981992270027, critical_value=1.959963984540054, standard_error=0.5, confidence_level=0.95)

### Intuición Singapur — "qué es 95% de confianza"

**No** es "el verdadero $\mu$ está con probabilidad 0{,}95 en este intervalo". El verdadero
$\mu$ es una constante (no es aleatorio). Lo aleatorio es la muestra y, por lo tanto, el
intervalo. La afirmación correcta es:

> **"Si repitiéramos este procedimiento muchas veces, el 95% de los intervalos producidos
> contendrían a $\mu$."**

El widget de abajo lo materializa: cada línea es un intervalo de una muestra distinta.


In [4]:
build_mean_ci_explorer(MeanCIExplorerInput(settings=settings))

## IC con $\sigma$ desconocido — la $t$ de Student

Cuando no conocemos $\sigma$ lo estimamos con $s$. Pero entonces el pivote
$T = (\bar{X} - \mu)/(s/\sqrt{n})$ tiene **más cola** que la Normal — distribuye como
$t_{n-1}$. Para $n$ grande la $t$ converge a la Normal estándar.


In [5]:
build_confidence_interval_for_mean_unknown_variance(
    MeanUnknownVarianceInput(
        sample_mean=12.0,
        sample_standard_deviation=3.0,
        sample_size=36,
        confidence_level=0.95,
    )
)

MeanConfidenceInterval(point_estimate=12.0, lower_bound=10.98494603587483, upper_bound=13.01505396412517, margin_of_error=1.0150539641251715, critical_value=2.030107928250343, standard_error=0.5, confidence_level=0.95)

## IC para proporción

En una encuesta de $n = 400$ personas, $200$ dicen "sí". $\hat{p} = 0{,}50$.
El IC de Wald al 95% queda:

$$ \hat{p} \pm z\,\sqrt{\frac{\hat{p}(1-\hat{p})}{n}}. $$


In [6]:
build_confidence_interval_for_proportion(ProportionInput(successes=200, sample_size=400, confidence_level=0.95))

ProportionConfidenceInterval(point_estimate=0.5, lower_bound=0.45100090038649865, upper_bound=0.5489990996135014, margin_of_error=0.04899909961350135, critical_value=1.959963984540054, standard_error=0.025, confidence_level=0.95)

## IC para la varianza — la $\chi^2$

Si $X_1, \dots, X_n \sim \mathcal{N}(\mu, \sigma^2)$, entonces
$\dfrac{(n-1)S^2}{\sigma^2} \sim \chi^2_{n-1}$.

**Intuición Singapur.** La $\chi^2$ es una **suma de cuadrados** de normales estándar.
Las desviaciones cuadráticas son no negativas: la $\chi^2$ vive en $[0, \infty)$.


In [7]:
build_confidence_interval_for_variance(VarianceInput(sample_variance=9.0, sample_size=36, confidence_level=0.95))

VarianceConfidenceInterval(point_estimate=9.0, lower_bound=5.920679969062422, upper_bound=15.314027530089101, lower_critical_value=20.56937663074498, upper_critical_value=53.20334854205644, degrees_of_freedom=35, confidence_level=0.95)

## Test de hipótesis para la media

Queremos chequear si la media de servicio es realmente 11 minutos (lo que dice el manual).
Observamos $\bar{x} = 12$, $s = 3$, $n = 36$.

- $H_0: \mu = 11$
- $H_1: \mu \ne 11$
- $\alpha = 0{,}05$

Estadístico: $T = (\bar{X} - 11)/(s/\sqrt{n})$.


In [8]:
mean_test_result = test_one_sample_mean(
    OneSampleMeanTestInput(
        sample_mean=12.0,
        sample_standard_deviation=3.0,
        sample_size=36,
        null_mean=11.0,
        alternative=Alternative.TWO_SIDED,
        significance_level=0.05,
    )
)
mean_test_result

OneSampleMeanTestResult(test_statistic=2.0, degrees_of_freedom=35, p_value=0.05330765186319679, reject_null=False, alternative=<Alternative.TWO_SIDED: 'two_sided'>)

### Conexión IC ↔ Test

Un test bilateral al nivel $\alpha$ rechaza $H_0: \mu = \mu_0$ **sí y sólo sí** $\mu_0$
**no cae** dentro del IC al nivel $1 - \alpha$ construido sobre la misma muestra.
Es el mismo objeto, visto desde dos ángulos.


## Bootstrap — IC sin supuestos paramétricos

Cuando no queremos asumir Normalidad ni conocer $\sigma$, podemos **remuestrear**: tomar
muchas muestras con reemplazo del conjunto observado y mirar la distribución de las
medias bootstrap. El IC sale de sus percentiles 2{,}5 y 97{,}5.


In [9]:
rng = np.random.default_rng(settings.random_seed)
synthetic_sample = Observations.validate(pd.DataFrame({"value": rng.normal(loc=12.0, scale=3.0, size=36)}))
bootstrap_result = bootstrap_mean(BootstrapInput(observations=synthetic_sample, replicates=3_000, settings=settings))
chart_bootstrap_distribution(BootstrapDistributionChartInput(bootstrap_result=bootstrap_result, settings=settings))

alt.LayerChart(...)

## Tamaño muestral para una precisión dada

Si queremos un margen de error $\le 1$ minuto con $95\%$ de confianza y sabemos que
$\sigma = 3$:

$$ n \ge \left(\frac{z\,\sigma}{E}\right)^2 = (1{,}96 \cdot 3 / 1)^2 \approx 35 $$


In [10]:
sample_size_for_mean(
    SampleSizeForMeanInput(
        population_standard_deviation=3.0,
        margin_of_error=1.0,
        confidence_level=0.95,
    )
)

SampleSizeResult(required_sample_size=35, critical_value=1.959963984540054, margin_of_error=1.0, confidence_level=0.95)

## Ejercicio 1 — IC contiene el verdadero $\mu$

Construir un IC al 95% para $\mu$ con $\bar{x} = 12$, $\sigma = 3$, $n = 36$, y
verificar que contenga el valor de control $\mu_0 = 11{,}5$.


In [11]:
interval = build_confidence_interval_for_mean_known_variance(
    MeanKnownVarianceInput(
        sample_mean=12.0,
        population_standard_deviation=3.0,
        sample_size=36,
        confidence_level=0.95,
    )
)
verify_interval_contains(
    IntervalContainsInput(
        lower_bound=interval.lower_bound,
        upper_bound=interval.upper_bound,
        target_value=11.5,
    )
)

VerificationResult(passed=True, message='OK — el intervalo [11.0200, 12.9800] contiene 11.5000')

## Ejercicio 2 — Tamaño muestral para una proporción

Queremos estimar la intención de voto con un margen de error de $\pm 3\%$ al $95\%$ de
confianza. Usamos $\hat{p} = 0{,}5$ (peor caso).

$$ n \ge \left(\frac{1{,}96}{0{,}03}\right)^2 \cdot 0{,}25 \approx 1067 $$


In [12]:
expected_size = sample_size_for_proportion(
    SampleSizeForProportionInput(
        estimated_proportion=0.5,
        margin_of_error=0.03,
        confidence_level=0.95,
    )
).required_sample_size

student_answer = 1067.0
verify_numeric_answer(
    NumericAnswerInput(
        student_answer=student_answer,
        expected_answer=float(expected_size),
        absolute_tolerance=1.0,
    )
)

VerificationResult(passed=True, message='OK — esperado 1068.000000')

## Para llevarse

- Un IC describe el **procedimiento**, no la muestra particular.
- $\sigma$ desconocido → $t$ de Student → colas más anchas que la Normal.
- Test e IC son **dual**: rechazar $H_0$ equivale a que $\mu_0$ caiga fuera del IC.
- Bootstrap evita la mayoría de los supuestos y se apoya en la **misma muestra observada**.
- El tamaño muestral crece como $1/E^2$: para reducir el margen a la mitad, hace falta $4\times$ más datos.
